# Math Standards Alignment Evaluator

The **Math Standards Alignment Evaluator** determines whether a K–12 math assessment question aligns to a specific academic standard, evaluated at the learning-component (LC) level.

Given a question, a standard code (e.g. `3.MD.C.7.d`), and a jurisdiction, it:
1. Resolves the standard to its Knowledge Graph UUID
2. Fetches the standard's learning components (LCs) from the Learning Commons Knowledge Graph
3. Sends all LCs to the LLM in a single batched call, getting a Yes/No alignment + reasoning per LC
4. Returns a structured result: per-LC decisions, aligned count, total count

This notebook loads `config.json`, `input_schema.json`, and `output_schema.json` from this directory — the same source-of-truth assets the TypeScript SDK reads. Running it validates that the prompts and KG integration work end-to-end.

In [ ]:
%pip install -qU anthropic requests

In [ ]:
import os
import json
import hashlib
import getpass
from pathlib import Path
import requests
import anthropic
import pprint as pp

In [ ]:
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

if "KG_API_KEY" not in os.environ:
    os.environ["KG_API_KEY"] = getpass.getpass("Enter your Learning Commons KG API key: ")

In [ ]:
# -------------------------------------------------------------------------
# Load source-of-truth assets: config.json + every prompt file declared
# in config.steps[*].prompt.messages
# -------------------------------------------------------------------------
# The canonical evaluator definition lives in evals/academic-standards-alignment/mathematics/math-standards-alignment.
# All consumers (this notebook, TypeScript SDK) read these same files.

ASSETS_DIR = Path(".")

with open(ASSETS_DIR / "config.json") as f:
    CONFIG = json.load(f)

with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)

with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

with open(ASSETS_DIR / "fixtures.json") as f:
    FIXTURES = json.load(f)

# Load prompt files declared in config and verify sha256 checksums.
# This is drift tripwire #1: if a prompt file is edited without updating
# the config sha256, this cell raises — preventing silent prompt/config divergence.
PROMPTS = {}
step = next(s for s in CONFIG["steps"] if s["id"] == "evaluate_math_standards_alignment")
for msg in step["prompt"]["messages"]:
    src = msg["source_path"]
    text = (ASSETS_DIR / src).read_text()
    if "sha256" in msg:
        actual = hashlib.sha256(text.encode()).hexdigest()
        assert actual == msg["sha256"], f"Prompt drift detected in {src}: expected {msg['sha256']}, got {actual}"
    PROMPTS[msg["role"]] = text

MODEL_NAME = step["model"]["name"]
TEMPERATURE = step["generation"]["temperature"]
KG_BASE_URL = "https://api.learningcommons.org/knowledge-graph/v0"

print(f"Loaded config: evaluator={CONFIG['evaluator']['id']}")
print(f"Model: {MODEL_NAME}, temperature={TEMPERATURE}")
print(f"Prompt roles loaded: {list(PROMPTS.keys())}")

In [ ]:
# -------------------------------------------------------------------------
# Knowledge Graph helpers
# (implements the preprocessing steps declared in config.json)
# -------------------------------------------------------------------------

def _kg_headers():
    return {"x-api-key": os.environ["KG_API_KEY"]}

def get_standard_uuid(statement_code: str, jurisdiction: str = "Multi-State") -> str:
    """Preprocessing step 1: resolve statementCode → caseIdentifierUUID."""
    resp = requests.get(
        f"{KG_BASE_URL}/academic-standards/search",
        headers=_kg_headers(),
        params={"statementCode": statement_code, "jurisdiction": jurisdiction, "limit": 1},
    )
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"Standard not found: {statement_code!r} in jurisdiction {jurisdiction!r}")
    return data[0]["caseIdentifierUUID"]

def get_learning_components(uuid: str) -> list[dict]:
    """Preprocessing step 2: fetch all LCs for a standard UUID (follows cursor pagination)."""
    lcs = []
    cursor = None
    while True:
        params = {"limit": 100}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(
            f"{KG_BASE_URL}/academic-standards/{uuid}/learning-components",
            headers=_kg_headers(),
            params=params,
        )
        resp.raise_for_status()
        body = resp.json()
        lcs.extend(
            {"identifier": lc["identifier"], "description": lc["description"]}
            for lc in body.get("data", [])
            if lc.get("description") and lc.get("identifier")
        )
        pagination = body.get("pagination", {})
        if not pagination.get("hasMore"):
            break
        cursor = pagination.get("nextCursor")
        if not cursor:
            raise ValueError(f"KG returned hasMore=True but no nextCursor for UUID {uuid}")
    return lcs

def format_lcs_for_prompt(lcs: list[dict]) -> str:
    """Formats LCs as the numbered list the user prompt expects."""
    return "\n".join(f"{i+1}. [{lc['identifier']}] {lc['description']}" for i, lc in enumerate(lcs))

In [ ]:
# -------------------------------------------------------------------------
# Evaluator — single question × single standard
# -------------------------------------------------------------------------

client = anthropic.Anthropic()

def evaluate(question: str, statement_code: str, jurisdiction: str = "Multi-State") -> dict:
    """
    Evaluates whether `question` aligns to `statement_code` in `jurisdiction`.
    Implements the primitive evaluate() from the TypeScript SDK in pure Python.

    Always returns:
      { rendered_prompt, raw_text, formatted_output, usage }
    rendered_prompt/raw_text/usage are None when the standard has no LCs.
    """
    # Preprocessing
    uuid = get_standard_uuid(statement_code, jurisdiction)
    lcs = get_learning_components(uuid)

    if not lcs:
        return {
            "rendered_prompt": None,
            "raw_text": None,
            "formatted_output": {
                "statementCode": statement_code,
                "learningComponents": [],
                "alignedCount": 0,
                "totalCount": 0,
            },
            "usage": None,
        }

    lc_text = format_lcs_for_prompt(lcs)

    # Render prompts
    user_prompt = (
        PROMPTS["user"]
        .replace("{question}", question)
        .replace("{learning_components}", lc_text)
        .replace("{n}", str(len(lcs)))
    )

    rendered_prompt = {
        "system": PROMPTS["system"],
        "user": user_prompt,
    }

    # LLM call
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        system=PROMPTS["system"],
        messages=[{"role": "user", "content": user_prompt}],
        temperature=TEMPERATURE,
    )

    raw_text = response.content[0].text
    usage = {"input_tokens": response.usage.input_tokens, "output_tokens": response.usage.output_tokens}

    # Strip markdown fences in case the model wraps its JSON output
    clean = raw_text.strip()
    if clean.startswith("```"):
        clean = clean[clean.index("\n") + 1:] if "\n" in clean else clean[3:]
        if clean.rstrip().endswith("```"):
            clean = clean.rstrip()[:-3].rstrip()

    # Parse structured output. user.txt asks for a bare JSON array (one
    # entry per learning component), not an {"evaluations": [...]} wrapper.
    evaluations = json.loads(clean)

    if len(evaluations) != len(lcs):
        print(f"Warning: LLM returned {len(evaluations)} evaluations but {len(lcs)} LCs were fetched for {statement_code}")

    # Build LC results — match back to KG data by lc_id
    lc_by_id = {lc["identifier"]: lc for lc in lcs}
    lc_results = []
    for ev in evaluations:
        lc = lc_by_id.get(ev["lc_id"])
        if lc is None:
            print(f"Warning: lc_id {ev['lc_id']!r} not found in fetched LCs — description will be missing")
            lc = {}
        aligned = ev["answer"] == "Yes"
        lc_results.append({
            "description": lc.get("description", ev["lc_id"]),
            "reasoning": ev["reasoning"],
            "aligned": aligned,
            "feedback": None if aligned else ev.get("feedback"),
        })

    aligned_count = sum(1 for lc in lc_results if lc["aligned"])

    formatted_output = {
        "statementCode": statement_code,
        "learningComponents": lc_results,
        "alignedCount": aligned_count,
        "totalCount": len(lcs),  # total fetched from KG, independent of LLM response length
    }

    return {
        "rendered_prompt": rendered_prompt,
        "raw_text": raw_text,
        "formatted_output": formatted_output,
        "usage": usage,
    }

In [ ]:
# -------------------------------------------------------------------------
# Sample run — L-shaped playground vs 3.MD.C.7.d
# -------------------------------------------------------------------------

sample_question = (
    "A playground is shaped like an L. One part is a rectangle that is 8 feet long and 3 feet wide. "
    "Attached to it is another rectangle that is 4 feet long and 2 feet wide, with no overlap. "
    "What is the total area of the playground in square feet?"
)

result = evaluate(sample_question, "3.MD.C.7.d", jurisdiction="Multi-State")

In [ ]:
print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
if result["rendered_prompt"]:
    pp.pprint(result["rendered_prompt"])
else:
    print("  (no LLM call — standard has no learning components)")

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(result["raw_text"] or "(no LLM call)")

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
out = result["formatted_output"]
print(f"  statementCode : {out['statementCode']}")
print(f"  alignedCount  : {out['alignedCount']} / {out['totalCount']}")
for lc in out["learningComponents"]:
    mark = "\u2713" if lc["aligned"] else "\u2717"
    print(f"  {mark} {lc['description']}")
    print(f"    reasoning: {lc['reasoning']}")
    if lc["feedback"]:
        print(f"    feedback:  {lc['feedback']}")

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(result["usage"])

In [ ]:
# -------------------------------------------------------------------------
# Sniff-test runner: load fixtures.json and check predictions against expected
# -------------------------------------------------------------------------
# Fixtures follow the schema in input_schema.json + output_schema.json.
# Each case has:
#   - id, description
#   - input: { question, statementCode, jurisdiction }  — runtime evaluator inputs
#   - expected: { aligned }                             — ground-truth label
#
# expected.aligned = (alignedCount > 0)

print(f"Running {len(FIXTURES)} fixtures...\n")

passed = 0
failed = 0

for fixture in FIXTURES:
    inp = fixture["input"]
    expected_aligned = fixture["expected"]["aligned"]

    r = evaluate(inp["question"], inp["statementCode"], inp.get("jurisdiction", "Multi-State"))
    out = r["formatted_output"]
    actual_aligned = out["alignedCount"] > 0

    ok = actual_aligned == expected_aligned
    status = "PASS" if ok else "FAIL"
    if ok:
        passed += 1
    else:
        failed += 1

    print(f"[{status}] fixture {fixture['id']}: {fixture['description']}")
    print(f"       standard={inp['statementCode']}  aligned={actual_aligned} (expected={expected_aligned})")
    print(f"       alignedCount={out['alignedCount']}/{out['totalCount']}")
    print()

print(f"Results: {passed}/{len(FIXTURES)} passed, {failed} failed")
assert failed == 0, f"{failed} fixture(s) failed"